# Appendix A8: all models get misaligned

Reproduces Figure A8 ("All models we studied get misaligned when finetuned
on wrong advice") of `Unequal_influence.pdf` - a byproduct of
Figure 5's per-model baseline training. No new
training jobs beyond what Figure 5 already runs.

**Prerequisite**:

```bash
uv run snakemake figure5 base_models
```

`base_models` evaluates each model before fine-tuning, for the paper's dashed
"Pre-finetune rate" reference lines.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS = Path("../../results")
OUTPUT_DIR = RESULTS / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MISALIGNED_THRESHOLD = 3
DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}

MODEL_LABELS = {
    "olmo": "OLMo 3 7B",
    "qwen2.5-1.5b": "Qwen 2.5 1B",
    "qwen2.5-3b": "Qwen 2.5 3B",
    "qwen2.5-7b": "Qwen 2.5 7B",
    "qwen2.5-14b": "Qwen 2.5 14B",
    "qwen3-4b": "Qwen 3 4B",
    "qwen3-8b": "Qwen 3 8B",
    "qwen3-14b": "Qwen 3 14B",
    "llama3.2-1b": "Llama 3.2 1B",
    "llama3.2-3b": "Llama 3.2 3B",
    "llama3.1-8b": "Llama 3.1 8B",
}
MODEL_ORDER = list(MODEL_LABELS)

PRETRAIN_RATES = {path.parent.name: 100 * (pd.read_csv(path)["aligned"] < MISALIGNED_THRESHOLD).mean()
                  for path in (RESULTS / "base").glob("*/answers.csv")}


## Load every model's baseline

Every model's unfiltered runs from `figure5.csv`, averaged over seeds.


In [ ]:
def load_all_model_baselines():
    rates = pd.read_csv(RESULTS / "figures" / "figure5.csv")
    baselines = rates[rates["subset"] == "full"]
    return baselines.groupby(["dataset", "model"], as_index=False)["misaligned_pct"].mean()


## Figure A8

In [ ]:
def plot_all_models_misaligned(datasets=("auto", "career", "edu"), pretrain_rates=PRETRAIN_RATES):
    rates = load_all_model_baselines()
    present = [model for model in MODEL_ORDER if model in rates["model"].unique()]

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(datasets))
    width = 0.8 / len(present)
    palette = plt.cm.tab20(np.linspace(0, 1, len(present)))
    for i, model in enumerate(present):
        values = [rates[(rates["dataset"] == d) & (rates["model"] == model)]["misaligned_pct"].mean() for d in datasets]
        offsets = x + (i - (len(present) - 1) / 2) * width
        ax.bar(offsets, values, width=width * 0.9, color=palette[i], label=MODEL_LABELS[model])
        if model in pretrain_rates:
            for xi in offsets:
                ax.hlines(pretrain_rates[model], xi - width * 0.45, xi + width * 0.45, color="black", linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels([DATASET_LABELS.get(d, d) for d in datasets])
    ax.set_ylabel("Misaligned completions (%)")
    ax.set_xlabel("Dataset")
    ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)
    fig.tight_layout()
    return fig


fig = plot_all_models_misaligned()
fig.savefig(OUTPUT_DIR / "figure_a8_all_models_misaligned.png", dpi=200, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "figure_a8_all_models_misaligned.pdf", bbox_inches="tight")
print(f"Saved to {OUTPUT_DIR / 'figure_a8_all_models_misaligned.png'}")
